# Droneport Destekli İHA Görevleri İçin Karar Destek ve Optimizasyon Sistemi

## Çalışmanın Amacı

Bu Notebook'ta, önceki aşamada geliştirilen ve kaydedilen makine öğrenmesi modeli kullanılarak İHA görevlerinin enerji ve batarya açısından uygunluğunu değerlendiren bir karar destek sistemi geliştirilecektir.

Sistem; görev mesafesi, ortalama uçuş hızı, operasyon süresi, faydalı yük ağırlığı, rüzgâr hızı ve sıcaklık değerlerini kullanarak gerekli enerji miktarını tahmin edecektir. Tahmin sonucu, başlangıç batarya seviyesi ve batarya sağlık oranıyla birlikte değerlendirilerek görev durumu aşağıdaki sınıflardan biriyle ifade edilecektir:

- **Uygun**
- **Riskli**
- **Uygun Değil**

Riskli veya uygun olmayan görevler için gerekli başlangıç batarya seviyesi hesaplanacak ve görev koşullarının iyileştirilmesine yönelik alternatif öneriler oluşturulacaktır.

Bu çalışma sentetik verilerle geliştirilmiş bir kavram kanıtlama uygulamasıdır. Kullanılan batarya kapasitesi, güvenlik rezervi ve görev sınırları gerçek bir ticari ürünün teknik özelliklerini temsil etmemektedir. Sistemin gerçek uçuş görevlerinde kullanılabilmesi için gerçek cihaz verileriyle doğrulanması ve ilgili güvenlik testlerinden geçirilmesi gerekmektedir.


In [1]:
# Dosya işlemleri, model yükleme ve veri analizi için gerekli kütüphaneler içe aktarılıyor.
from pathlib import Path
import hashlib
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn

from google.colab import drive
from IPython.display import display

# Gereksiz uyarıların Notebook çıktısında gösterilmesi engelleniyor.
warnings.filterwarnings("ignore")

# Google Drive daha önce bağlanmadıysa bağlantı kuruluyor.
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

# Proje klasörleri tanımlanıyor.
PROJE_KLASORU = Path(
    "/content/drive/MyDrive/Droneport_Enerji_Optimizasyonu"
)

VERI_KLASORU = PROJE_KLASORU / "veri"
MODEL_KLASORU = PROJE_KLASORU / "modeller"
SONUC_KLASORU = PROJE_KLASORU / "sonuclar"

# Nihai model ve metadata dosyalarının yolları tanımlanıyor.
NIHAI_MODEL_DOSYASI = (
    MODEL_KLASORU / "optimize_gradyan_artirma_modeli.joblib"
)

MODEL_METADATA_DOSYASI = (
    MODEL_KLASORU / "model_metadata.json"
)

# Önceki Notebook'ta kaydedilen model dosyasının referans SHA-256 değeri tanımlanıyor.
BEKLENEN_MODEL_SHA256 = (
    "bd776e7034c7fdaa83c5a80f59c0b46836828f1b9a3a9f28d6bce6d20ed75fa5"
)

# Gerekli dosyaların mevcut olup olmadığı kontrol ediliyor.
if not NIHAI_MODEL_DOSYASI.exists():
    raise FileNotFoundError(
        f"Model dosyası bulunamadı: {NIHAI_MODEL_DOSYASI}"
    )

if not MODEL_METADATA_DOSYASI.exists():
    raise FileNotFoundError(
        f"Metadata dosyası bulunamadı: {MODEL_METADATA_DOSYASI}"
    )

# Model dosyasının güncel SHA-256 değeri hesaplanıyor.
model_sha256_hesaplayici = hashlib.sha256()

with open(NIHAI_MODEL_DOSYASI, "rb") as model_dosyasi:
    for veri_blogu in iter(
        lambda: model_dosyasi.read(8192),
        b""
    ):
        model_sha256_hesaplayici.update(veri_blogu)

hesaplanan_model_sha256 = (
    model_sha256_hesaplayici.hexdigest()
)

# Hesaplanan SHA-256 değeri referans değerle karşılaştırılıyor.
model_butunlugu_dogrulandi = (
    hesaplanan_model_sha256
    == BEKLENEN_MODEL_SHA256
)

if not model_butunlugu_dogrulandi:
    raise ValueError(
        "Model dosyasının SHA-256 değeri beklenen değerle eşleşmedi."
    )

# Model metadata bilgileri yükleniyor.
with open(
    MODEL_METADATA_DOSYASI,
    "r",
    encoding="utf-8"
) as metadata_dosyasi:
    model_metadata = json.load(
        metadata_dosyasi
    )

# Metadata içindeki model özeti de dosya değeriyle karşılaştırılıyor.
metadata_model_sha256_dogrulandi = (
    model_metadata["model_sha256"]
    == hesaplanan_model_sha256
)

if not metadata_model_sha256_dogrulandi:
    raise ValueError(
        "Metadata dosyasındaki model özeti gerçek model dosyasıyla eşleşmedi."
    )

# Doğrulanan nihai model belleğe yükleniyor.
nihai_model = joblib.load(
    NIHAI_MODEL_DOSYASI
)

# Modelin beklediği özellik sırası metadata dosyasından alınıyor.
model_ozellikleri = model_metadata[
    "model_ozellikleri"
]

# Model ve metadata bilgileri görüntüleniyor.
print("NİHAİ MODEL BAŞARIYLA YÜKLENDİ")
print("=" * 85)
print(
    f"Model adı                    : "
    f"{model_metadata['model_adi']}"
)
print(
    f"Model sınıfı                 : "
    f"{model_metadata['model_sinifi']}"
)
print(
    f"Scikit-learn sürümü          : "
    f"{sklearn.__version__}"
)
print(
    f"Model özellik sayısı         : "
    f"{len(model_ozellikleri)}"
)
print(
    f"Model bütünlüğü doğrulandı   : "
    f"{model_butunlugu_dogrulandi}"
)
print(
    f"Metadata özeti doğrulandı    : "
    f"{metadata_model_sha256_dogrulandi}"
)
print(
    f"Kaydedilen test MAE değeri   : "
    f"{model_metadata['test_mae_wh']:.4f} Wh"
)
print(
    f"Kaydedilen test RMSE değeri  : "
    f"{model_metadata['test_rmse_wh']:.4f} Wh"
)
print(
    f"Kaydedilen test R² değeri    : "
    f"{model_metadata['test_r2']:.4f}"
)
print(
    f"Model SHA-256                : "
    f"{hesaplanan_model_sha256}"
)

print("\nMODELİN BEKLEDİĞİ ÖZELLİK SIRASI")
print("=" * 85)

for sira, ozellik in enumerate(
    model_ozellikleri,
    start=1
):
    print(f"{sira}. {ozellik}")


Mounted at /content/drive
NİHAİ MODEL BAŞARIYLA YÜKLENDİ
Model adı                    : Optimize Gradyan Artırma Regresyonu
Model sınıfı                 : GradientBoostingRegressor
Scikit-learn sürümü          : 1.6.1
Model özellik sayısı         : 8
Model bütünlüğü doğrulandı   : True
Metadata özeti doğrulandı    : True
Kaydedilen test MAE değeri   : 8.9065 Wh
Kaydedilen test RMSE değeri  : 12.3851 Wh
Kaydedilen test R² değeri    : 0.9758
Model SHA-256                : bd776e7034c7fdaa83c5a80f59c0b46836828f1b9a3a9f28d6bce6d20ed75fa5

MODELİN BEKLEDİĞİ ÖZELLİK SIRASI
1. gorev_mesafesi_km
2. ortalama_ucus_hizi_kmh
3. operasyon_suresi_dk
4. toplam_gorev_suresi_dk
5. faydali_yuk_kg
6. ruzgar_hizi_mps
7. ortam_sicakligi_c
8. batarya_sicakligi_c


## 1. Karar Destek Sistemi Varsayımlarının Tanımlanması

Karar destek sisteminde nominal batarya kapasitesi 600 Wh ve görev sonunda korunması hedeflenen güvenli batarya rezervi %20 olarak kabul edilmiştir. Batarya sağlık oranı, nominal kapasitenin ne kadarının etkin biçimde kullanılabildiğini temsil etmektedir.

Etkin batarya kapasitesi aşağıdaki ilişkiyle hesaplanmıştır:

\[
\text{Etkin Kapasite} =
\text{Nominal Kapasite}
\times
\frac{\text{Batarya Sağlığı}}{100}
\]

Görev başlangıcında kullanılabilir enerji ise etkin kapasite ve başlangıç doluluk oranı kullanılarak belirlenmiştir:

\[
\text{Kullanılabilir Enerji} =
\text{Etkin Kapasite}
\times
\frac{\text{Başlangıç Batarya Seviyesi}}{100}
\]

Görev durumunun belirlenmesinde tahmin edilen enerji ihtiyacından sonra kalacak batarya oranı kullanılmıştır:

- **Uygun:** Kalan batarya oranı %20 veya daha yüksek,
- **Riskli:** Kalan batarya oranı %10 ile %20 arasında,
- **Uygun Değil:** Kalan batarya oranı %10'un altında veya görev enerjisi mevcut enerjiden fazla.

Bu eşikler gerçek bir ürünün üretici güvenlik sınırları değildir. Kavram kanıtlama çalışmasının karar mantığını göstermek amacıyla belirlenen simülasyon varsayımlarıdır.

Makine öğrenmesi modelleri eğitim verisinin dışında güvenilir biçimde genelleme garantisi vermez. Bu nedenle görev girdileri modelin eğitim aralıklarıyla karşılaştırılacaktır. Eğitim aralığının dışındaki değerlerde enerji tahmini üretilebilse bile sonuç güvenilir görev onayı olarak kullanılmayacak ve kapsam dışı uyarısı verilecektir.


In [2]:
# Karar destek sisteminde kullanılacak batarya varsayımları tanımlanıyor.
NOMINAL_BATARYA_KAPASITESI_WH = 600.0
GUVENLI_BATARYA_REZERVI_YUZDE = 20.0
KRITIK_BATARYA_REZERVI_YUZDE = 10.0

# Batarya durumu için kabul edilen simülasyon sınırları tanımlanıyor.
BATARYA_SAGLIGI_ALT_SINIRI = 70.0
BATARYA_SAGLIGI_UST_SINIRI = 100.0

BASLANGIC_BATARYA_ALT_SINIRI = 0.0
BASLANGIC_BATARYA_UST_SINIRI = 100.0

# Modelin eğitiminde kullanılan görev değişkenlerinin güvenilir giriş aralıkları tanımlanıyor.
# Bu sınırlar gerçek bir ürünün teknik sınırları değil, sentetik eğitim verisinin kapsamıdır.
model_girdi_sinirlari = {
    "gorev_mesafesi_km": {
        "alt_sinir": 1.0,
        "ust_sinir": 20.0,
        "birim": "km",
        "aciklama": "Toplam görev mesafesi"
    },
    "ortalama_ucus_hizi_kmh": {
        "alt_sinir": 20.0,
        "ust_sinir": 50.0,
        "birim": "km/sa",
        "aciklama": "Ortalama uçuş hızı"
    },
    "operasyon_suresi_dk": {
        "alt_sinir": 0.0,
        "ust_sinir": 12.0,
        "birim": "dk",
        "aciklama": "Operasyon süresi"
    },
    "toplam_gorev_suresi_dk": {
        "alt_sinir": 2.92,
        "ust_sinir": 55.59,
        "birim": "dk",
        "aciklama": "Toplam görev süresi"
    },
    "faydali_yuk_kg": {
        "alt_sinir": 0.0,
        "ust_sinir": 2.0,
        "birim": "kg",
        "aciklama": "Faydalı yük ağırlığı"
    },
    "ruzgar_hizi_mps": {
        "alt_sinir": 0.0,
        "ust_sinir": 14.0,
        "birim": "m/sn",
        "aciklama": "Ortalama rüzgâr hızı"
    },
    "ortam_sicakligi_c": {
        "alt_sinir": -10.0,
        "ust_sinir": 40.0,
        "birim": "°C",
        "aciklama": "Ortam sıcaklığı"
    },
    "batarya_sicakligi_c": {
        "alt_sinir": -5.0,
        "ust_sinir": 45.0,
        "birim": "°C",
        "aciklama": "Batarya sıcaklığı"
    }
}

# Model giriş sınırları okunabilir bir tabloya dönüştürülüyor.
model_girdi_sinirlari_tablosu = pd.DataFrame([
    {
        "Değişken": degisken,
        "Açıklama": bilgiler["aciklama"],
        "Alt Sınır": bilgiler["alt_sinir"],
        "Üst Sınır": bilgiler["ust_sinir"],
        "Birim": bilgiler["birim"]
    }
    for degisken, bilgiler
    in model_girdi_sinirlari.items()
])

# Karar eşikleri ayrı bir tabloda hazırlanıyor.
karar_esikleri_tablosu = pd.DataFrame({
    "Görev Durumu": [
        "Uygun",
        "Riskli",
        "Uygun Değil"
    ],
    "Kalan Batarya Koşulu": [
        "%20 veya daha fazla",
        "%10 ile %20 arasında",
        "%10'dan az veya enerji yetersiz"
    ]
})

print("KARAR DESTEK SİSTEMİ AYARLARI TANIMLANDI")
print("=" * 85)
print(
    f"Nominal batarya kapasitesi : "
    f"{NOMINAL_BATARYA_KAPASITESI_WH:.0f} Wh"
)
print(
    f"Güvenli batarya rezervi    : "
    f"%{GUVENLI_BATARYA_REZERVI_YUZDE:.0f}"
)
print(
    f"Kritik batarya rezervi     : "
    f"%{KRITIK_BATARYA_REZERVI_YUZDE:.0f}"
)
print(
    "Batarya sağlığı aralığı    : "
    f"%{BATARYA_SAGLIGI_ALT_SINIRI:.0f}–"
    f"%{BATARYA_SAGLIGI_UST_SINIRI:.0f}"
)
print(
    "Başlangıç batarya aralığı  : "
    f"%{BASLANGIC_BATARYA_ALT_SINIRI:.0f}–"
    f"%{BASLANGIC_BATARYA_UST_SINIRI:.0f}"
)

print("\nMODEL GİRDİLERİNİN GÜVENİLİR ARALIKLARI")
print("=" * 85)
display(model_girdi_sinirlari_tablosu)

print("\nGÖREV DURUMU KARAR EŞİKLERİ")
print("=" * 85)
display(karar_esikleri_tablosu)


KARAR DESTEK SİSTEMİ AYARLARI TANIMLANDI
Nominal batarya kapasitesi : 600 Wh
Güvenli batarya rezervi    : %20
Kritik batarya rezervi     : %10
Batarya sağlığı aralığı    : %70–%100
Başlangıç batarya aralığı  : %0–%100

MODEL GİRDİLERİNİN GÜVENİLİR ARALIKLARI


,Değişken,Açıklama,Alt Sınır,Üst Sınır,Birim
0,gorev_mesafesi_km,Toplam görev mesafesi,1.00,20.00,km
1,ortalama_ucus_hizi_kmh,Ortalama uçuş hızı,20.00,50.00,km/sa
2,operasyon_suresi_dk,Operasyon süresi,0.00,12.00,dk
3,toplam_gorev_suresi_dk,Toplam görev süresi,2.92,55.59,dk
4,faydali_yuk_kg,Faydalı yük ağırlığı,0.00,2.00,kg
5,ruzgar_hizi_mps,Ortalama rüzgâr hızı,0.00,14.00,m/sn
6,ortam_sicakligi_c,Ortam sıcaklığı,-10.00,40.00,°C
7,batarya_sicakligi_c,Batarya sıcaklığı,-5.00,45.00,°C



GÖREV DURUMU KARAR EŞİKLERİ


,Görev Durumu,Kalan Batarya Koşulu
0,Uygun,%20 veya daha fazla
1,Riskli,%10 ile %20 arasında
2,Uygun Değil,%10'dan az veya enerji yetersiz


## 2. Görev Girdilerinin Doğrulanması

Karar destek sistemine girilen değerler enerji tahmini yapılmadan önce doğrulanmıştır. Negatif görev mesafesi, sıfır veya negatif uçuş hızı, negatif faydalı yük ve %100'ün üzerindeki batarya seviyesi gibi fiziksel veya mantıksal olarak geçersiz değerlerde hesaplama durdurulmaktadır.

Fiziksel olarak mümkün olmakla birlikte modelin eğitim kapsamının dışında kalan değerlerde ise tahmin işlemi tamamen engellenmemektedir. Bunun yerine ilgili değişken için kapsam dışı uyarısı oluşturulmaktadır. Böylece modelin ürettiği sayısal sonuç ile bu sonucun güvenilirlik durumu birbirinden ayrılmıştır.

Toplam görev süresi kullanıcıdan bağımsız bir giriş olarak alınmamıştır. Görev mesafesi, ortalama uçuş hızı ve operasyon süresi kullanılarak aşağıdaki ilişkiyle hesaplanmıştır:

\[
\text{Toplam Görev Süresi} =
\left(
\frac{\text{Görev Mesafesi}}
{\text{Ortalama Uçuş Hızı}}
\times 60
\right)
+
\text{Operasyon Süresi}
\]

Bu yöntemle birbiriyle uyumsuz mesafe, hız ve süre değerlerinin sisteme girilmesi önlenmiştir.


In [3]:
# Görev girdilerinin sayısal ve sonlu olup olmadığını kontrol eden yardımcı fonksiyon tanımlanıyor.
def sayisal_degeri_dogrula(
    degisken_adi,
    deger
):
    """
    Girilen değeri ondalıklı sayıya dönüştürür ve
    sonlu bir sayısal değer olup olmadığını kontrol eder.
    """

    try:
        sayisal_deger = float(deger)
    except (TypeError, ValueError):
        raise ValueError(
            f"{degisken_adi} sayısal bir değer olmalıdır."
        )

    if not np.isfinite(sayisal_deger):
        raise ValueError(
            f"{degisken_adi} sonlu bir sayısal değer olmalıdır."
        )

    return sayisal_deger


# Görev girdilerini doğrulayan ve model için gerekli veri satırını oluşturan ana fonksiyon tanımlanıyor.
def gorev_girdilerini_hazirla(
    gorev_mesafesi_km,
    ortalama_ucus_hizi_kmh,
    operasyon_suresi_dk,
    faydali_yuk_kg,
    ruzgar_hizi_mps,
    ortam_sicakligi_c,
    batarya_sicakligi_c,
    baslangic_batarya_yuzde,
    batarya_sagligi_yuzde
):
    """
    Görev ve batarya girdilerini doğrular, toplam görev süresini
    hesaplar ve model tahmini için tek satırlık veri çerçevesi oluşturur.
    """

    # Bütün girişler sayısal değerlere dönüştürülüyor.
    gorev_mesafesi_km = sayisal_degeri_dogrula(
        "Görev mesafesi",
        gorev_mesafesi_km
    )

    ortalama_ucus_hizi_kmh = sayisal_degeri_dogrula(
        "Ortalama uçuş hızı",
        ortalama_ucus_hizi_kmh
    )

    operasyon_suresi_dk = sayisal_degeri_dogrula(
        "Operasyon süresi",
        operasyon_suresi_dk
    )

    faydali_yuk_kg = sayisal_degeri_dogrula(
        "Faydalı yük ağırlığı",
        faydali_yuk_kg
    )

    ruzgar_hizi_mps = sayisal_degeri_dogrula(
        "Rüzgâr hızı",
        ruzgar_hizi_mps
    )

    ortam_sicakligi_c = sayisal_degeri_dogrula(
        "Ortam sıcaklığı",
        ortam_sicakligi_c
    )

    batarya_sicakligi_c = sayisal_degeri_dogrula(
        "Batarya sıcaklığı",
        batarya_sicakligi_c
    )

    baslangic_batarya_yuzde = sayisal_degeri_dogrula(
        "Başlangıç batarya seviyesi",
        baslangic_batarya_yuzde
    )

    batarya_sagligi_yuzde = sayisal_degeri_dogrula(
        "Batarya sağlığı",
        batarya_sagligi_yuzde
    )

    # Fiziksel ve mantıksal olarak geçersiz girişler kontrol ediliyor.
    if gorev_mesafesi_km <= 0:
        raise ValueError(
            "Görev mesafesi sıfırdan büyük olmalıdır."
        )

    if ortalama_ucus_hizi_kmh <= 0:
        raise ValueError(
            "Ortalama uçuş hızı sıfırdan büyük olmalıdır."
        )

    if operasyon_suresi_dk < 0:
        raise ValueError(
            "Operasyon süresi negatif olamaz."
        )

    if faydali_yuk_kg < 0:
        raise ValueError(
            "Faydalı yük ağırlığı negatif olamaz."
        )

    if ruzgar_hizi_mps < 0:
        raise ValueError(
            "Rüzgâr hızı negatif olamaz."
        )

    if not (
        BASLANGIC_BATARYA_ALT_SINIRI
        <= baslangic_batarya_yuzde
        <= BASLANGIC_BATARYA_UST_SINIRI
    ):
        raise ValueError(
            "Başlangıç batarya seviyesi %0 ile %100 arasında olmalıdır."
        )

    if not (
        BATARYA_SAGLIGI_ALT_SINIRI
        <= batarya_sagligi_yuzde
        <= BATARYA_SAGLIGI_UST_SINIRI
    ):
        raise ValueError(
            "Batarya sağlığı simülasyon kapsamında "
            "%70 ile %100 arasında olmalıdır."
        )

    # Toplam görev süresi mesafe, hız ve operasyon süresinden hesaplanıyor.
    toplam_gorev_suresi_dk = (
        (
            gorev_mesafesi_km
            / ortalama_ucus_hizi_kmh
        )
        * 60.0
        + operasyon_suresi_dk
    )

    # Modelin beklediği özellikler doğru sırayla hazırlanıyor.
    model_girdisi = pd.DataFrame(
        [
            {
                "gorev_mesafesi_km": gorev_mesafesi_km,
                "ortalama_ucus_hizi_kmh": ortalama_ucus_hizi_kmh,
                "operasyon_suresi_dk": operasyon_suresi_dk,
                "toplam_gorev_suresi_dk": toplam_gorev_suresi_dk,
                "faydali_yuk_kg": faydali_yuk_kg,
                "ruzgar_hizi_mps": ruzgar_hizi_mps,
                "ortam_sicakligi_c": ortam_sicakligi_c,
                "batarya_sicakligi_c": batarya_sicakligi_c
            }
        ],
        columns=model_ozellikleri
    )

    # Model eğitim aralıklarının dışında kalan değerler için uyarılar hazırlanıyor.
    kapsam_uyarilari = []

    for degisken in model_ozellikleri:
        deger = float(
            model_girdisi.loc[0, degisken]
        )

        sinir_bilgisi = model_girdi_sinirlari[
            degisken
        ]

        alt_sinir = sinir_bilgisi[
            "alt_sinir"
        ]

        ust_sinir = sinir_bilgisi[
            "ust_sinir"
        ]

        if deger < alt_sinir or deger > ust_sinir:
            kapsam_uyarilari.append(
                (
                    f"{sinir_bilgisi['aciklama']} değeri "
                    f"({deger:.2f} {sinir_bilgisi['birim']}) "
                    f"model kapsamının dışındadır. "
                    f"Beklenen aralık: "
                    f"{alt_sinir:.2f}–{ust_sinir:.2f} "
                    f"{sinir_bilgisi['birim']}."
                )
            )

    # Doğrulanan batarya bilgileri ayrı bir sözlükte saklanıyor.
    batarya_bilgileri = {
        "baslangic_batarya_yuzde": (
            baslangic_batarya_yuzde
        ),
        "batarya_sagligi_yuzde": (
            batarya_sagligi_yuzde
        )
    }

    return {
        "model_girdisi": model_girdisi,
        "batarya_bilgileri": batarya_bilgileri,
        "kapsam_uyarilari": kapsam_uyarilari,
        "model_kapsaminda": (
            len(kapsam_uyarilari) == 0
        )
    }


# Girdi doğrulama fonksiyonu olağan bir görev senaryosuyla test ediliyor.
ornek_girdi_kontrolu = gorev_girdilerini_hazirla(
    gorev_mesafesi_km=8.0,
    ortalama_ucus_hizi_kmh=32.0,
    operasyon_suresi_dk=4.0,
    faydali_yuk_kg=0.7,
    ruzgar_hizi_mps=5.0,
    ortam_sicakligi_c=20.0,
    batarya_sicakligi_c=24.0,
    baslangic_batarya_yuzde=90.0,
    batarya_sagligi_yuzde=92.0
)

print("GÖREV GİRDİ DOĞRULAMA TESTİ")
print("=" * 85)
print(
    f"Model kapsamında: "
    f"{ornek_girdi_kontrolu['model_kapsaminda']}"
)
print(
    f"Kapsam uyarısı sayısı: "
    f"{len(ornek_girdi_kontrolu['kapsam_uyarilari'])}"
)
print(
    "Başlangıç batarya seviyesi: "
    f"%{ornek_girdi_kontrolu['batarya_bilgileri']['baslangic_batarya_yuzde']:.2f}"
)
print(
    "Batarya sağlığı            : "
    f"%{ornek_girdi_kontrolu['batarya_bilgileri']['batarya_sagligi_yuzde']:.2f}"
)

print("\nMODEL İÇİN HAZIRLANAN GİRDİ")
print("=" * 85)
display(
    ornek_girdi_kontrolu[
        "model_girdisi"
    ].round(2)
)


GÖREV GİRDİ DOĞRULAMA TESTİ
Model kapsamında: True
Kapsam uyarısı sayısı: 0
Başlangıç batarya seviyesi: %90.00
Batarya sağlığı            : %92.00

MODEL İÇİN HAZIRLANAN GİRDİ


,gorev_mesafesi_km,ortalama_ucus_hizi_kmh,operasyon_suresi_dk,toplam_gorev_suresi_dk,faydali_yuk_kg,ruzgar_hizi_mps,ortam_sicakligi_c,batarya_sicakligi_c
0,8.0,32.0,4.0,19.0,0.7,5.0,20.0,24.0


## 3. Enerji Tahmini ve Batarya Uygunluk Değerlendirmesi

Doğrulanan görev girdileri nihai makine öğrenmesi modeline verilerek görevin tahmini enerji ihtiyacı hesaplanmıştır. Karar destek sisteminin yalnızca nokta tahminine bağlı kalmaması amacıyla, modelin bağımsız test kümesindeki RMSE değeri ek belirsizlik payı olarak kullanılmıştır.

\[
\text{Karar Enerjisi} =
\text{Tahmin Edilen Enerji}
+
\text{Test RMSE}
\]

Bu ek değer istatistiksel bir güvenlik garantisi değildir. Model hatalarına karşı daha temkinli bir karar üretmek amacıyla kullanılan kavram kanıtlama varsayımıdır.

Batarya hesaplamalarında aşağıdaki değerler belirlenmiştir:

- Etkin batarya kapasitesi,
- Başlangıçta kullanılabilir enerji,
- Görev sonrasında tahmini kalan enerji,
- Kalan batarya oranı,
- Güvenli rezerv için gerekli en düşük başlangıç batarya seviyesi,
- Mevcut enerji ile karar enerjisi arasındaki enerji fazlası veya açığı.

Görev durumu aşağıdaki kurallarla belirlenmiştir:

- Karar enerjisi mevcut enerjiden fazlaysa **Uygun Değil**,
- Kalan batarya oranı %10'un altındaysa **Uygun Değil**,
- Kalan batarya oranı %10 ile %20 arasındaysa **Riskli**,
- Kalan batarya oranı %20 veya daha yüksekse **Uygun**.

Model kapsamı dışındaki girdiler için batarya durumu ayrıca hesaplanabilmektedir. Ancak bu durumda sonuç güvenilir görev onayı olarak kullanılmamakta ve manuel inceleme uyarısı oluşturulmaktadır.


In [4]:
# Nihai modelin test RMSE değeri karar belirsizlik payı olarak tanımlanıyor.
MODEL_BELIRSIZLIK_PAYI_WH = float(
    model_metadata["test_rmse_wh"]
)


# Görevin enerji ve batarya uygunluğunu değerlendiren ana fonksiyon tanımlanıyor.
def gorevi_degerlendir(
    gorev_mesafesi_km,
    ortalama_ucus_hizi_kmh,
    operasyon_suresi_dk,
    faydali_yuk_kg,
    ruzgar_hizi_mps,
    ortam_sicakligi_c,
    batarya_sicakligi_c,
    baslangic_batarya_yuzde,
    batarya_sagligi_yuzde
):
    """
    Görev girdilerini doğrular, enerji ihtiyacını tahmin eder
    ve batarya uygunluk durumunu belirler.
    """

    # Görev girdileri doğrulanıyor ve model satırı hazırlanıyor.
    hazirlanan_girdiler = gorev_girdilerini_hazirla(
        gorev_mesafesi_km=gorev_mesafesi_km,
        ortalama_ucus_hizi_kmh=ortalama_ucus_hizi_kmh,
        operasyon_suresi_dk=operasyon_suresi_dk,
        faydali_yuk_kg=faydali_yuk_kg,
        ruzgar_hizi_mps=ruzgar_hizi_mps,
        ortam_sicakligi_c=ortam_sicakligi_c,
        batarya_sicakligi_c=batarya_sicakligi_c,
        baslangic_batarya_yuzde=baslangic_batarya_yuzde,
        batarya_sagligi_yuzde=batarya_sagligi_yuzde
    )

    model_girdisi = hazirlanan_girdiler[
        "model_girdisi"
    ]

    batarya_bilgileri = hazirlanan_girdiler[
        "batarya_bilgileri"
    ]

    # Makine öğrenmesi modeliyle enerji ihtiyacı tahmin ediliyor.
    tahmini_enerji_wh = float(
        nihai_model.predict(
            model_girdisi
        )[0]
    )

    # Model hatasına karşı daha temkinli karar enerjisi hesaplanıyor.
    karar_enerjisi_wh = (
        tahmini_enerji_wh
        + MODEL_BELIRSIZLIK_PAYI_WH
    )

    baslangic_batarya_yuzde = batarya_bilgileri[
        "baslangic_batarya_yuzde"
    ]

    batarya_sagligi_yuzde = batarya_bilgileri[
        "batarya_sagligi_yuzde"
    ]

    # Batarya sağlık oranına göre etkin kapasite hesaplanıyor.
    etkin_batarya_kapasitesi_wh = (
        NOMINAL_BATARYA_KAPASITESI_WH
        * batarya_sagligi_yuzde
        / 100.0
    )

    # Görev başlangıcında kullanılabilir enerji hesaplanıyor.
    kullanilabilir_enerji_wh = (
        etkin_batarya_kapasitesi_wh
        * baslangic_batarya_yuzde
        / 100.0
    )

    # Nokta tahminine ve karar enerjisine göre kalan enerji hesaplanıyor.
    tahmini_kalan_enerji_wh = (
        kullanilabilir_enerji_wh
        - tahmini_enerji_wh
    )

    karar_sonrasi_kalan_enerji_wh = (
        kullanilabilir_enerji_wh
        - karar_enerjisi_wh
    )

    # Karar sonrası kalan batarya oranı hesaplanıyor.
    karar_sonrasi_kalan_batarya_yuzde = (
        karar_sonrasi_kalan_enerji_wh
        / etkin_batarya_kapasitesi_wh
        * 100.0
    )

    # Gösterim için negatif batarya oranı sıfırda sınırlandırılıyor.
    gosterilen_kalan_batarya_yuzde = max(
        0.0,
        karar_sonrasi_kalan_batarya_yuzde
    )

    # Güvenli rezervle göreve başlanabilmesi için gerekli en düşük şarj hesaplanıyor.
    gerekli_en_dusuk_baslangic_yuzde = (
        karar_enerjisi_wh
        / etkin_batarya_kapasitesi_wh
        * 100.0
        + GUVENLI_BATARYA_REZERVI_YUZDE
    )

    # Mevcut enerji ile karar enerjisi arasındaki fark hesaplanıyor.
    enerji_dengesi_wh = (
        kullanilabilir_enerji_wh
        - karar_enerjisi_wh
    )

    # Görev durumu batarya rezervine göre belirleniyor.
    if enerji_dengesi_wh < 0:
        gorev_durumu = "Uygun Değil"
        karar_gerekcesi = (
            "Mevcut batarya enerjisi görevin temkinli enerji "
            "ihtiyacını karşılamamaktadır."
        )

    elif (
        karar_sonrasi_kalan_batarya_yuzde
        < KRITIK_BATARYA_REZERVI_YUZDE
    ):
        gorev_durumu = "Uygun Değil"
        karar_gerekcesi = (
            "Görev sonrasında kalması beklenen batarya oranı "
            "kritik rezerv sınırının altındadır."
        )

    elif (
        karar_sonrasi_kalan_batarya_yuzde
        < GUVENLI_BATARYA_REZERVI_YUZDE
    ):
        gorev_durumu = "Riskli"
        karar_gerekcesi = (
            "Görev tamamlanabilir görünmektedir ancak güvenli "
            "batarya rezervi sağlanamamaktadır."
        )

    else:
        gorev_durumu = "Uygun"
        karar_gerekcesi = (
            "Tahmini enerji ihtiyacı karşılanmakta ve güvenli "
            "batarya rezervi korunmaktadır."
        )

    # Model kapsamı dışındaysa otomatik onay verilmemesi sağlanıyor.
    if hazirlanan_girdiler["model_kapsaminda"]:
        degerlendirme_guveni = "Model Kapsamında"
        nihai_onay = gorev_durumu
    else:
        degerlendirme_guveni = "Model Kapsamı Dışında"
        nihai_onay = "Manuel İnceleme Gerekli"

    return {
        "gorev_durumu": gorev_durumu,
        "nihai_onay": nihai_onay,
        "karar_gerekcesi": karar_gerekcesi,
        "degerlendirme_guveni": degerlendirme_guveni,
        "model_kapsaminda": hazirlanan_girdiler[
            "model_kapsaminda"
        ],
        "kapsam_uyarilari": hazirlanan_girdiler[
            "kapsam_uyarilari"
        ],
        "model_girdisi": model_girdisi.copy(),
        "tahmini_enerji_wh": tahmini_enerji_wh,
        "model_belirsizlik_payi_wh": (
            MODEL_BELIRSIZLIK_PAYI_WH
        ),
        "karar_enerjisi_wh": karar_enerjisi_wh,
        "etkin_batarya_kapasitesi_wh": (
            etkin_batarya_kapasitesi_wh
        ),
        "kullanilabilir_enerji_wh": (
            kullanilabilir_enerji_wh
        ),
        "tahmini_kalan_enerji_wh": (
            tahmini_kalan_enerji_wh
        ),
        "karar_sonrasi_kalan_enerji_wh": (
            karar_sonrasi_kalan_enerji_wh
        ),
        "kalan_batarya_yuzde": (
            gosterilen_kalan_batarya_yuzde
        ),
        "ham_kalan_batarya_yuzde": (
            karar_sonrasi_kalan_batarya_yuzde
        ),
        "gerekli_en_dusuk_baslangic_yuzde": (
            gerekli_en_dusuk_baslangic_yuzde
        ),
        "enerji_dengesi_wh": enerji_dengesi_wh,
        "baslangic_batarya_yuzde": (
            baslangic_batarya_yuzde
        ),
        "batarya_sagligi_yuzde": (
            batarya_sagligi_yuzde
        )
    }


# Ana değerlendirme fonksiyonu önceki olağan görev senaryosuyla test ediliyor.
ornek_gorev_sonucu = gorevi_degerlendir(
    gorev_mesafesi_km=8.0,
    ortalama_ucus_hizi_kmh=32.0,
    operasyon_suresi_dk=4.0,
    faydali_yuk_kg=0.7,
    ruzgar_hizi_mps=5.0,
    ortam_sicakligi_c=20.0,
    batarya_sicakligi_c=24.0,
    baslangic_batarya_yuzde=90.0,
    batarya_sagligi_yuzde=92.0
)

# Örnek sonuç okunabilir bir tabloya dönüştürülüyor.
ornek_sonuc_tablosu = pd.DataFrame({
    "Değerlendirme": [
        "Tahmini enerji ihtiyacı",
        "Model belirsizlik payı",
        "Karar enerjisi",
        "Etkin batarya kapasitesi",
        "Kullanılabilir enerji",
        "Görev sonrası kalan enerji",
        "Görev sonrası kalan batarya",
        "Gerekli en düşük başlangıç bataryası",
        "Enerji dengesi",
        "Görev durumu",
        "Nihai onay",
        "Değerlendirme güveni"
    ],
    "Sonuç": [
        f"{ornek_gorev_sonucu['tahmini_enerji_wh']:.2f} Wh",
        f"{ornek_gorev_sonucu['model_belirsizlik_payi_wh']:.2f} Wh",
        f"{ornek_gorev_sonucu['karar_enerjisi_wh']:.2f} Wh",
        f"{ornek_gorev_sonucu['etkin_batarya_kapasitesi_wh']:.2f} Wh",
        f"{ornek_gorev_sonucu['kullanilabilir_enerji_wh']:.2f} Wh",
        f"{ornek_gorev_sonucu['karar_sonrasi_kalan_enerji_wh']:.2f} Wh",
        f"%{ornek_gorev_sonucu['kalan_batarya_yuzde']:.2f}",
        f"%{ornek_gorev_sonucu['gerekli_en_dusuk_baslangic_yuzde']:.2f}",
        f"{ornek_gorev_sonucu['enerji_dengesi_wh']:.2f} Wh",
        ornek_gorev_sonucu["gorev_durumu"],
        ornek_gorev_sonucu["nihai_onay"],
        ornek_gorev_sonucu["degerlendirme_guveni"]
    ]
})

print("ÖRNEK GÖREV ENERJİ VE BATARYA DEĞERLENDİRMESİ")
print("=" * 90)
display(ornek_sonuc_tablosu)

print("\nKARAR GEREKÇESİ")
print("=" * 90)
print(
    ornek_gorev_sonucu[
        "karar_gerekcesi"
    ]
)


ÖRNEK GÖREV ENERJİ VE BATARYA DEĞERLENDİRMESİ


,Değerlendirme,Sonuç
0,Tahmini enerji ihtiyacı,176.72 Wh
1,Model belirsizlik payı,12.39 Wh
2,Karar enerjisi,189.10 Wh
3,Etkin batarya kapasitesi,552.00 Wh
4,Kullanılabilir enerji,496.80 Wh
5,Görev sonrası kalan enerji,307.70 Wh
6,Görev sonrası kalan batarya,%55.74
7,Gerekli en düşük başlangıç bataryası,%54.26
8,Enerji dengesi,307.70 Wh
9,Görev durumu,Uygun



KARAR GEREKÇESİ
Tahmini enerji ihtiyacı karşılanmakta ve güvenli batarya rezervi korunmaktadır.


## 4. Senaryo Tabanlı İyileştirme Önerilerinin Oluşturulması

Riskli veya uygun olmayan görevler için görev koşullarını iyileştiren alternatif senaryolar araştırılmıştır. Her değişkenin etkisinin açık biçimde görülebilmesi amacıyla değişiklikler birbirinden bağımsız olarak değerlendirilmiştir.

Aşağıdaki alternatifler incelenmiştir:

- Başlangıç batarya seviyesinin artırılması,
- Faydalı yük ağırlığının azaltılması,
- Daha düşük rüzgâr hızının beklenmesi,
- Görev mesafesinin azaltılması,
- Daha sağlıklı bir batarya kullanılması.

Başlangıç batarya seviyesi için gerekli değer doğrudan enerji hesabından elde edilmiştir. Diğer değişkenlerde ise mevcut değere en yakın güvenli koşulun bulunması amacıyla kontrollü adımlarla senaryo araması uygulanmıştır.

Her alternatifte yalnızca incelenen değişken değiştirilmiş, diğer görev koşulları sabit tutulmuştur. Bu nedenle oluşturulan öneriler birbirinden bağımsız seçeneklerdir ve aynı anda uygulanmaları zorunlu değildir.

Modelin eğitim kapsamı dışına çıkan alternatifler öneri olarak kabul edilmemiştir. Öneriler gerçek uçuş güvenliği onayı değil, geliştirilen kavram kanıtlama sisteminin senaryo tabanlı karar desteği olarak değerlendirilmiştir.


In [5]:
# Riskli veya uygun olmayan görevler için
# bağımsız iyileştirme seçenekleri üreten fonksiyon tanımlanıyor.
def iyilestirme_onerileri_uret(
    gorev_mesafesi_km,
    ortalama_ucus_hizi_kmh,
    operasyon_suresi_dk,
    faydali_yuk_kg,
    ruzgar_hizi_mps,
    ortam_sicakligi_c,
    batarya_sicakligi_c,
    baslangic_batarya_yuzde,
    batarya_sagligi_yuzde
):
    """
    Görev koşullarını ayrı ayrı değiştirerek görevi
    uygun duruma getirebilecek en yakın seçenekleri araştırır.
    """

    # Mevcut görev durumu hesaplanıyor.
    mevcut_sonuc = gorevi_degerlendir(
        gorev_mesafesi_km=gorev_mesafesi_km,
        ortalama_ucus_hizi_kmh=ortalama_ucus_hizi_kmh,
        operasyon_suresi_dk=operasyon_suresi_dk,
        faydali_yuk_kg=faydali_yuk_kg,
        ruzgar_hizi_mps=ruzgar_hizi_mps,
        ortam_sicakligi_c=ortam_sicakligi_c,
        batarya_sicakligi_c=batarya_sicakligi_c,
        baslangic_batarya_yuzde=baslangic_batarya_yuzde,
        batarya_sagligi_yuzde=batarya_sagligi_yuzde
    )

    oneriler = []

    # Görev zaten uygunsa değişiklik önerilmesine gerek duyulmuyor.
    if (
        mevcut_sonuc["gorev_durumu"] == "Uygun"
        and mevcut_sonuc["model_kapsaminda"]
    ):
        return {
            "mevcut_sonuc": mevcut_sonuc,
            "oneriler": [],
            "aciklama": (
                "Görev mevcut koşullarda uygundur. "
                "Ek iyileştirme önerisi oluşturulmamıştır."
            )
        }

    # Model kapsamı dışındaki başlangıç senaryosu için
    # otomatik optimizasyon yapılmıyor.
    if not mevcut_sonuc["model_kapsaminda"]:
        return {
            "mevcut_sonuc": mevcut_sonuc,
            "oneriler": [],
            "aciklama": (
                "Görev girdileri model kapsamı dışında olduğu için "
                "otomatik iyileştirme önerisi oluşturulmamıştır."
            )
        }

    # 1. Güvenli rezerv için gerekli başlangıç batarya seviyesi değerlendiriliyor.
    gerekli_baslangic_yuzde = mevcut_sonuc[
        "gerekli_en_dusuk_baslangic_yuzde"
    ]

    if (
        gerekli_baslangic_yuzde
        <= 100.0
        and gerekli_baslangic_yuzde
        > baslangic_batarya_yuzde
    ):
        onerilen_baslangic_yuzde = min(
            100.0,
            np.ceil(
                gerekli_baslangic_yuzde
            )
        )

        batarya_senaryosu = gorevi_degerlendir(
            gorev_mesafesi_km=gorev_mesafesi_km,
            ortalama_ucus_hizi_kmh=ortalama_ucus_hizi_kmh,
            operasyon_suresi_dk=operasyon_suresi_dk,
            faydali_yuk_kg=faydali_yuk_kg,
            ruzgar_hizi_mps=ruzgar_hizi_mps,
            ortam_sicakligi_c=ortam_sicakligi_c,
            batarya_sicakligi_c=batarya_sicakligi_c,
            baslangic_batarya_yuzde=onerilen_baslangic_yuzde,
            batarya_sagligi_yuzde=batarya_sagligi_yuzde
        )

        if batarya_senaryosu["gorev_durumu"] == "Uygun":
            oneriler.append({
                "Öneri Türü": "Başlangıç bataryasını artırma",
                "Mevcut Değer": (
                    f"%{baslangic_batarya_yuzde:.2f}"
                ),
                "Önerilen Değer": (
                    f"%{onerilen_baslangic_yuzde:.2f}"
                ),
                "Beklenen Kalan Batarya": (
                    f"%{batarya_senaryosu['kalan_batarya_yuzde']:.2f}"
                ),
                "Beklenen Durum": (
                    batarya_senaryosu["gorev_durumu"]
                )
            })

    # 2. Faydalı yük, mevcut değerden başlayarak 0,10 kg adımlarla azaltılıyor.
    yuk_senaryosu_bulundu = False

    for aday_yuk in np.arange(
        faydali_yuk_kg - 0.10,
        -0.01,
        -0.10
    ):
        aday_yuk = max(
            0.0,
            round(float(aday_yuk), 2)
        )

        yuk_senaryosu = gorevi_degerlendir(
            gorev_mesafesi_km=gorev_mesafesi_km,
            ortalama_ucus_hizi_kmh=ortalama_ucus_hizi_kmh,
            operasyon_suresi_dk=operasyon_suresi_dk,
            faydali_yuk_kg=aday_yuk,
            ruzgar_hizi_mps=ruzgar_hizi_mps,
            ortam_sicakligi_c=ortam_sicakligi_c,
            batarya_sicakligi_c=batarya_sicakligi_c,
            baslangic_batarya_yuzde=baslangic_batarya_yuzde,
            batarya_sagligi_yuzde=batarya_sagligi_yuzde
        )

        if (
            yuk_senaryosu["gorev_durumu"] == "Uygun"
            and yuk_senaryosu["model_kapsaminda"]
        ):
            oneriler.append({
                "Öneri Türü": "Faydalı yükü azaltma",
                "Mevcut Değer": (
                    f"{faydali_yuk_kg:.2f} kg"
                ),
                "Önerilen Değer": (
                    f"{aday_yuk:.2f} kg"
                ),
                "Beklenen Kalan Batarya": (
                    f"%{yuk_senaryosu['kalan_batarya_yuzde']:.2f}"
                ),
                "Beklenen Durum": (
                    yuk_senaryosu["gorev_durumu"]
                )
            })

            yuk_senaryosu_bulundu = True
            break

    # Değişkenin kullanıldığı, fakat her durumda çözüm bulunamayabileceği belirtiliyor.
    _ = yuk_senaryosu_bulundu

    # 3. Rüzgâr hızı 0,50 m/sn adımlarla azaltılıyor.
    for aday_ruzgar in np.arange(
        ruzgar_hizi_mps - 0.50,
        -0.01,
        -0.50
    ):
        aday_ruzgar = max(
            0.0,
            round(float(aday_ruzgar), 2)
        )

        ruzgar_senaryosu = gorevi_degerlendir(
            gorev_mesafesi_km=gorev_mesafesi_km,
            ortalama_ucus_hizi_kmh=ortalama_ucus_hizi_kmh,
            operasyon_suresi_dk=operasyon_suresi_dk,
            faydali_yuk_kg=faydali_yuk_kg,
            ruzgar_hizi_mps=aday_ruzgar,
            ortam_sicakligi_c=ortam_sicakligi_c,
            batarya_sicakligi_c=batarya_sicakligi_c,
            baslangic_batarya_yuzde=baslangic_batarya_yuzde,
            batarya_sagligi_yuzde=batarya_sagligi_yuzde
        )

        if (
            ruzgar_senaryosu["gorev_durumu"] == "Uygun"
            and ruzgar_senaryosu["model_kapsaminda"]
        ):
            oneriler.append({
                "Öneri Türü": "Daha düşük rüzgârı bekleme",
                "Mevcut Değer": (
                    f"{ruzgar_hizi_mps:.2f} m/sn"
                ),
                "Önerilen Değer": (
                    f"{aday_ruzgar:.2f} m/sn"
                ),
                "Beklenen Kalan Batarya": (
                    f"%{ruzgar_senaryosu['kalan_batarya_yuzde']:.2f}"
                ),
                "Beklenen Durum": (
                    ruzgar_senaryosu["gorev_durumu"]
                )
            })
            break

    # 4. Görev mesafesi 0,50 km adımlarla azaltılıyor.
    for aday_mesafe in np.arange(
        gorev_mesafesi_km - 0.50,
        0.99,
        -0.50
    ):
        aday_mesafe = max(
            1.0,
            round(float(aday_mesafe), 2)
        )

        mesafe_senaryosu = gorevi_degerlendir(
            gorev_mesafesi_km=aday_mesafe,
            ortalama_ucus_hizi_kmh=ortalama_ucus_hizi_kmh,
            operasyon_suresi_dk=operasyon_suresi_dk,
            faydali_yuk_kg=faydali_yuk_kg,
            ruzgar_hizi_mps=ruzgar_hizi_mps,
            ortam_sicakligi_c=ortam_sicakligi_c,
            batarya_sicakligi_c=batarya_sicakligi_c,
            baslangic_batarya_yuzde=baslangic_batarya_yuzde,
            batarya_sagligi_yuzde=batarya_sagligi_yuzde
        )

        if (
            mesafe_senaryosu["gorev_durumu"] == "Uygun"
            and mesafe_senaryosu["model_kapsaminda"]
        ):
            oneriler.append({
                "Öneri Türü": "Görev mesafesini azaltma",
                "Mevcut Değer": (
                    f"{gorev_mesafesi_km:.2f} km"
                ),
                "Önerilen Değer": (
                    f"{aday_mesafe:.2f} km"
                ),
                "Beklenen Kalan Batarya": (
                    f"%{mesafe_senaryosu['kalan_batarya_yuzde']:.2f}"
                ),
                "Beklenen Durum": (
                    mesafe_senaryosu["gorev_durumu"]
                )
            })
            break

    # 5. Batarya sağlığı %1 adımlarla artırılarak daha sağlıklı batarya seçeneği araştırılıyor.
    for aday_saglik in np.arange(
        np.floor(batarya_sagligi_yuzde) + 1.0,
        100.1,
        1.0
    ):
        aday_saglik = round(
            float(aday_saglik),
            2
        )

        saglik_senaryosu = gorevi_degerlendir(
            gorev_mesafesi_km=gorev_mesafesi_km,
            ortalama_ucus_hizi_kmh=ortalama_ucus_hizi_kmh,
            operasyon_suresi_dk=operasyon_suresi_dk,
            faydali_yuk_kg=faydali_yuk_kg,
            ruzgar_hizi_mps=ruzgar_hizi_mps,
            ortam_sicakligi_c=ortam_sicakligi_c,
            batarya_sicakligi_c=batarya_sicakligi_c,
            baslangic_batarya_yuzde=baslangic_batarya_yuzde,
            batarya_sagligi_yuzde=aday_saglik
        )

        if (
            saglik_senaryosu["gorev_durumu"] == "Uygun"
            and saglik_senaryosu["model_kapsaminda"]
        ):
            oneriler.append({
                "Öneri Türü": "Daha sağlıklı batarya kullanma",
                "Mevcut Değer": (
                    f"%{batarya_sagligi_yuzde:.2f}"
                ),
                "Önerilen Değer": (
                    f"%{aday_saglik:.2f}"
                ),
                "Beklenen Kalan Batarya": (
                    f"%{saglik_senaryosu['kalan_batarya_yuzde']:.2f}"
                ),
                "Beklenen Durum": (
                    saglik_senaryosu["gorev_durumu"]
                )
            })
            break

    if len(oneriler) == 0:
        aciklama = (
            "Değişkenlerin tek başına değiştirilmesiyle uygun bir "
            "senaryo bulunamamıştır. Birden fazla koşulun birlikte "
            "iyileştirilmesi veya görevin yeniden planlanması gerekir."
        )
    else:
        aciklama = (
            "Aşağıdaki seçenekler birbirinden bağımsız olarak "
            "hesaplanmıştır. Her satır alternatif bir görev koşulunu "
            "göstermektedir."
        )

    return {
        "mevcut_sonuc": mevcut_sonuc,
        "oneriler": oneriler,
        "aciklama": aciklama
    }


# Öneri sistemini değerlendirmek için zorlu bir görev senaryosu tanımlanıyor.
zor_gorev_onerileri = iyilestirme_onerileri_uret(
    gorev_mesafesi_km=14.0,
    ortalama_ucus_hizi_kmh=25.0,
    operasyon_suresi_dk=8.0,
    faydali_yuk_kg=1.5,
    ruzgar_hizi_mps=10.0,
    ortam_sicakligi_c=5.0,
    batarya_sicakligi_c=10.0,
    baslangic_batarya_yuzde=75.0,
    batarya_sagligi_yuzde=85.0
)

zor_gorev_sonucu = zor_gorev_onerileri[
    "mevcut_sonuc"
]

print("ZORLU GÖREV SENARYOSUNUN MEVCUT DURUMU")
print("=" * 95)
print(
    f"Tahmini enerji ihtiyacı : "
    f"{zor_gorev_sonucu['tahmini_enerji_wh']:.2f} Wh"
)
print(
    f"Karar enerjisi          : "
    f"{zor_gorev_sonucu['karar_enerjisi_wh']:.2f} Wh"
)
print(
    f"Kullanılabilir enerji   : "
    f"{zor_gorev_sonucu['kullanilabilir_enerji_wh']:.2f} Wh"
)
print(
    f"Kalan batarya oranı     : "
    f"%{zor_gorev_sonucu['kalan_batarya_yuzde']:.2f}"
)
print(
    f"Mevcut görev durumu     : "
    f"{zor_gorev_sonucu['gorev_durumu']}"
)
print(
    f"Gerekli başlangıç şarjı : "
    f"%{zor_gorev_sonucu['gerekli_en_dusuk_baslangic_yuzde']:.2f}"
)

print("\nİYİLEŞTİRME AÇIKLAMASI")
print("=" * 95)
print(
    zor_gorev_onerileri["aciklama"]
)

print("\nBAĞIMSIZ İYİLEŞTİRME ÖNERİLERİ")
print("=" * 95)

if len(zor_gorev_onerileri["oneriler"]) > 0:
    zor_gorev_oneri_tablosu = pd.DataFrame(
        zor_gorev_onerileri["oneriler"]
    )
    display(zor_gorev_oneri_tablosu)
else:
    print(
        "Tek değişkenli uygun bir alternatif bulunamadı."
    )


ZORLU GÖREV SENARYOSUNUN MEVCUT DURUMU
Tahmini enerji ihtiyacı : 544.61 Wh
Karar enerjisi          : 556.99 Wh
Kullanılabilir enerji   : 382.50 Wh
Kalan batarya oranı     : %0.00
Mevcut görev durumu     : Uygun Değil
Gerekli başlangıç şarjı : %129.21

İYİLEŞTİRME AÇIKLAMASI
Aşağıdaki seçenekler birbirinden bağımsız olarak hesaplanmıştır. Her satır alternatif bir görev koşulunu göstermektedir.

BAĞIMSIZ İYİLEŞTİRME ÖNERİLERİ


,Öneri Türü,Mevcut Değer,Önerilen Değer,Beklenen Kalan Batarya,Beklenen Durum
0,Görev mesafesini azaltma,14.00 km,3.50 km,%23.53,Uygun


## 5. Karar Destek Sisteminin Farklı Senaryolarla Test Edilmesi

Geliştirilen karar destek sisteminin farklı batarya ve görev koşullarında beklenen kararları üretip üretmediği dört ayrı senaryo kullanılarak test edilmiştir:

1. Güvenli rezervin üzerinde batarya bırakan uygun görev,
2. Görevi tamamlayabilen ancak güvenli rezervi sağlayamayan riskli görev,
3. Kritik rezervin altında kalan uygun olmayan görev,
4. Modelin eğitim kapsamı dışında kalan ve manuel inceleme gerektiren görev.

İlk üç senaryoda başlangıç batarya seviyesi, karar enerjisi sonrasında sırasıyla yaklaşık %30, %15 ve %5 batarya kalacak şekilde hesaplanmıştır. Bu yöntemle karar eşiklerinin doğru çalışıp çalışmadığı kontrollü olarak doğrulanmıştır.

Kapsam dışı senaryoda fiziksel olarak hesaplanabilir ancak eğitim verisinde bulunmayan bir görev mesafesi kullanılmıştır. Bu durumda sistemin otomatik görev onayı vermemesi beklenmiştir.


In [6]:
# Belirli bir kalan batarya oranını hedefleyen başlangıç şarjını
# hesaplayan yardımcı fonksiyon tanımlanıyor.
def hedef_rezerve_gore_baslangic_hesapla(
    gorev_parametreleri,
    batarya_sagligi_yuzde,
    hedef_kalan_batarya_yuzde
):
    """
    Görev enerjisi ve batarya sağlığına göre belirlenen rezervi
    bırakacak yaklaşık başlangıç batarya seviyesini hesaplar.
    """

    # Enerji tahmininin başlangıç batarya seviyesinden etkilenmemesi nedeniyle
    # geçici olarak %100 başlangıç seviyesi kullanılıyor.
    gecici_sonuc = gorevi_degerlendir(
        **gorev_parametreleri,
        baslangic_batarya_yuzde=100.0,
        batarya_sagligi_yuzde=batarya_sagligi_yuzde
    )

    etkin_kapasite_wh = gecici_sonuc[
        "etkin_batarya_kapasitesi_wh"
    ]

    karar_enerjisi_wh = gecici_sonuc[
        "karar_enerjisi_wh"
    ]

    gerekli_baslangic_yuzde = (
        karar_enerjisi_wh
        / etkin_kapasite_wh
        * 100.0
        + hedef_kalan_batarya_yuzde
    )

    return round(
        float(gerekli_baslangic_yuzde),
        2
    )


# Model kapsamında kalan üç farklı görev koşulu tanımlanıyor.
uygun_gorev_parametreleri = {
    "gorev_mesafesi_km": 7.0,
    "ortalama_ucus_hizi_kmh": 34.0,
    "operasyon_suresi_dk": 3.0,
    "faydali_yuk_kg": 0.5,
    "ruzgar_hizi_mps": 3.0,
    "ortam_sicakligi_c": 22.0,
    "batarya_sicakligi_c": 25.0
}

riskli_gorev_parametreleri = {
    "gorev_mesafesi_km": 10.0,
    "ortalama_ucus_hizi_kmh": 30.0,
    "operasyon_suresi_dk": 5.0,
    "faydali_yuk_kg": 0.9,
    "ruzgar_hizi_mps": 6.0,
    "ortam_sicakligi_c": 15.0,
    "batarya_sicakligi_c": 19.0
}

uygun_olmayan_gorev_parametreleri = {
    "gorev_mesafesi_km": 12.0,
    "ortalama_ucus_hizi_kmh": 28.0,
    "operasyon_suresi_dk": 6.0,
    "faydali_yuk_kg": 1.2,
    "ruzgar_hizi_mps": 8.0,
    "ortam_sicakligi_c": 8.0,
    "batarya_sicakligi_c": 13.0
}

# Senaryolarda kullanılacak batarya sağlık oranları tanımlanıyor.
uygun_batarya_sagligi = 95.0
riskli_batarya_sagligi = 90.0
uygun_olmayan_batarya_sagligi = 88.0

# Karar eşiklerini kontrollü biçimde test etmek amacıyla başlangıç
# batarya seviyeleri hedeflenen kalan rezervlere göre hesaplanıyor.
uygun_baslangic_bataryasi = (
    hedef_rezerve_gore_baslangic_hesapla(
        uygun_gorev_parametreleri,
        uygun_batarya_sagligi,
        hedef_kalan_batarya_yuzde=30.0
    )
)

riskli_baslangic_bataryasi = (
    hedef_rezerve_gore_baslangic_hesapla(
        riskli_gorev_parametreleri,
        riskli_batarya_sagligi,
        hedef_kalan_batarya_yuzde=15.0
    )
)

uygun_olmayan_baslangic_bataryasi = (
    hedef_rezerve_gore_baslangic_hesapla(
        uygun_olmayan_gorev_parametreleri,
        uygun_olmayan_batarya_sagligi,
        hedef_kalan_batarya_yuzde=5.0
    )
)

# Hesaplanan başlangıç seviyelerinin fiziksel olarak uygulanabilir olduğu doğrulanıyor.
hesaplanan_baslangic_degerleri = [
    uygun_baslangic_bataryasi,
    riskli_baslangic_bataryasi,
    uygun_olmayan_baslangic_bataryasi
]

if any(
    deger < 0.0 or deger > 100.0
    for deger in hesaplanan_baslangic_degerleri
):
    raise ValueError(
        "Test senaryolarından biri için hesaplanan başlangıç "
        "batarya seviyesi %0–%100 aralığının dışındadır."
    )

# Uygun görev senaryosu değerlendiriliyor.
uygun_sonuc = gorevi_degerlendir(
    **uygun_gorev_parametreleri,
    baslangic_batarya_yuzde=uygun_baslangic_bataryasi,
    batarya_sagligi_yuzde=uygun_batarya_sagligi
)

# Riskli görev senaryosu değerlendiriliyor.
riskli_sonuc = gorevi_degerlendir(
    **riskli_gorev_parametreleri,
    baslangic_batarya_yuzde=riskli_baslangic_bataryasi,
    batarya_sagligi_yuzde=riskli_batarya_sagligi
)

# Uygun olmayan görev senaryosu değerlendiriliyor.
uygun_olmayan_sonuc = gorevi_degerlendir(
    **uygun_olmayan_gorev_parametreleri,
    baslangic_batarya_yuzde=uygun_olmayan_baslangic_bataryasi,
    batarya_sagligi_yuzde=uygun_olmayan_batarya_sagligi
)

# Model kapsamı dışındaki bir görev senaryosu değerlendiriliyor.
kapsam_disi_sonuc = gorevi_degerlendir(
    gorev_mesafesi_km=24.0,
    ortalama_ucus_hizi_kmh=35.0,
    operasyon_suresi_dk=4.0,
    faydali_yuk_kg=0.8,
    ruzgar_hizi_mps=5.0,
    ortam_sicakligi_c=20.0,
    batarya_sicakligi_c=24.0,
    baslangic_batarya_yuzde=95.0,
    batarya_sagligi_yuzde=95.0
)

# Bütün senaryo sonuçları ortak bir tabloda hazırlanıyor.
senaryo_sonuclari = pd.DataFrame([
    {
        "Senaryo": "Uygun Görev",
        "Başlangıç Bataryası (%)": uygun_baslangic_bataryasi,
        "Tahmini Enerji (Wh)": uygun_sonuc["tahmini_enerji_wh"],
        "Karar Enerjisi (Wh)": uygun_sonuc["karar_enerjisi_wh"],
        "Kalan Batarya (%)": uygun_sonuc["kalan_batarya_yuzde"],
        "Görev Durumu": uygun_sonuc["gorev_durumu"],
        "Nihai Onay": uygun_sonuc["nihai_onay"],
        "Model Kapsamında": uygun_sonuc["model_kapsaminda"]
    },
    {
        "Senaryo": "Riskli Görev",
        "Başlangıç Bataryası (%)": riskli_baslangic_bataryasi,
        "Tahmini Enerji (Wh)": riskli_sonuc["tahmini_enerji_wh"],
        "Karar Enerjisi (Wh)": riskli_sonuc["karar_enerjisi_wh"],
        "Kalan Batarya (%)": riskli_sonuc["kalan_batarya_yuzde"],
        "Görev Durumu": riskli_sonuc["gorev_durumu"],
        "Nihai Onay": riskli_sonuc["nihai_onay"],
        "Model Kapsamında": riskli_sonuc["model_kapsaminda"]
    },
    {
        "Senaryo": "Uygun Olmayan Görev",
        "Başlangıç Bataryası (%)": (
            uygun_olmayan_baslangic_bataryasi
        ),
        "Tahmini Enerji (Wh)": (
            uygun_olmayan_sonuc["tahmini_enerji_wh"]
        ),
        "Karar Enerjisi (Wh)": (
            uygun_olmayan_sonuc["karar_enerjisi_wh"]
        ),
        "Kalan Batarya (%)": (
            uygun_olmayan_sonuc["kalan_batarya_yuzde"]
        ),
        "Görev Durumu": (
            uygun_olmayan_sonuc["gorev_durumu"]
        ),
        "Nihai Onay": (
            uygun_olmayan_sonuc["nihai_onay"]
        ),
        "Model Kapsamında": (
            uygun_olmayan_sonuc["model_kapsaminda"]
        )
    },
    {
        "Senaryo": "Model Kapsamı Dışı",
        "Başlangıç Bataryası (%)": 95.0,
        "Tahmini Enerji (Wh)": (
            kapsam_disi_sonuc["tahmini_enerji_wh"]
        ),
        "Karar Enerjisi (Wh)": (
            kapsam_disi_sonuc["karar_enerjisi_wh"]
        ),
        "Kalan Batarya (%)": (
            kapsam_disi_sonuc["kalan_batarya_yuzde"]
        ),
        "Görev Durumu": (
            kapsam_disi_sonuc["gorev_durumu"]
        ),
        "Nihai Onay": (
            kapsam_disi_sonuc["nihai_onay"]
        ),
        "Model Kapsamında": (
            kapsam_disi_sonuc["model_kapsaminda"]
        )
    }
])

# Sayısal değerler okunabilir biçimde yuvarlanıyor.
senaryo_sonuclari[
    [
        "Başlangıç Bataryası (%)",
        "Tahmini Enerji (Wh)",
        "Karar Enerjisi (Wh)",
        "Kalan Batarya (%)"
    ]
] = senaryo_sonuclari[
    [
        "Başlangıç Bataryası (%)",
        "Tahmini Enerji (Wh)",
        "Karar Enerjisi (Wh)",
        "Kalan Batarya (%)"
    ]
].round(2)

# Beklenen kararların üretilip üretilmediği kontrol ediliyor.
beklenen_kararlar = [
    "Uygun",
    "Riskli",
    "Uygun Değil",
    "Manuel İnceleme Gerekli"
]

gerceklesen_kararlar = [
    uygun_sonuc["nihai_onay"],
    riskli_sonuc["nihai_onay"],
    uygun_olmayan_sonuc["nihai_onay"],
    kapsam_disi_sonuc["nihai_onay"]
]

senaryo_testleri_basarili = (
    gerceklesen_kararlar
    == beklenen_kararlar
)

print("KARAR DESTEK SİSTEMİ SENARYO TESTLERİ")
print("=" * 105)
display(senaryo_sonuclari)

print("\nKAPSAM DIŞI SENARYO UYARILARI")
print("=" * 105)

for sira, uyari in enumerate(
    kapsam_disi_sonuc["kapsam_uyarilari"],
    start=1
):
    print(f"{sira}. {uyari}")

print("\nSENARYO TESTLERİ DOĞRULAMA SONUCU")
print("=" * 105)
print(
    f"Beklenen kararlar   : {beklenen_kararlar}"
)
print(
    f"Gerçekleşen kararlar: {gerceklesen_kararlar}"
)
print(
    f"Bütün senaryo testleri başarılı: "
    f"{senaryo_testleri_basarili}"
)


KARAR DESTEK SİSTEMİ SENARYO TESTLERİ


,Senaryo,Başlangıç Bataryası (%),Tahmini Enerji (Wh),Karar Enerjisi (Wh),Kalan Batarya (%),Görev Durumu,Nihai Onay,Model Kapsamında
0,Uygun Görev,55.66,133.86,146.24,30.00,Uygun,Uygun,True
1,Riskli Görev,64.30,253.85,266.23,15.00,Riskli,Riskli,True
2,Uygun Olmayan Görev,76.35,364.33,376.72,5.00,Uygun Değil,Uygun Değil,True
3,Model Kapsamı Dışı,95.00,396.08,408.46,23.34,Uygun,Manuel İnceleme Gerekli,False



KAPSAM DIŞI SENARYO UYARILARI
1. Toplam görev mesafesi değeri (24.00 km) model kapsamının dışındadır. Beklenen aralık: 1.00–20.00 km.

SENARYO TESTLERİ DOĞRULAMA SONUCU
Beklenen kararlar   : ['Uygun', 'Riskli', 'Uygun Değil', 'Manuel İnceleme Gerekli']
Gerçekleşen kararlar: ['Uygun', 'Riskli', 'Uygun Değil', 'Manuel İnceleme Gerekli']
Bütün senaryo testleri başarılı: True


## 6. Karar Destek Sistemi Sonuçlarının Kaydedilmesi

Karar destek sisteminde kullanılan batarya varsayımları, karar eşikleri, model giriş sınırları ve belirsizlik payı bir yapılandırma dosyasına kaydedilmiştir.


Uygun، riskli، uygun olmayan ve model kapsamı dışındaki görev senaryolarının sonuçları ayrı bir CSV dosyasında saklanmıştır. Zorlu görev için oluşturulan bağımsız iyileştirme önerileri de daha sonraki incelemelerde kullanılmak üzere kaydedilmiştir.

Kaydedilen sonuçlar yeniden yüklenerek satır sayıları ve senaryo kararları doğrulanmıştır. Böylece karar destek aşamasının sonuçları kalıcı hâle getirilmiş ve proje çıktılarının tekrar incelenebilmesi sağlanmıştır.


In [7]:
# Sonuç dosyalarının oluşturulması için gerekli kütüphane içe aktarılıyor.
import json

# Karar destek sistemi çıktı dosyalarının yolları tanımlanıyor.
KARAR_SISTEMI_AYAR_DOSYASI = (
    SONUC_KLASORU / "karar_destek_sistemi_ayarlari.json"
)

SENARYO_SONUCLARI_DOSYASI = (
    SONUC_KLASORU / "karar_destek_senaryo_sonuclari.csv"
)

IYILESTIRME_ONERILERI_DOSYASI = (
    SONUC_KLASORU / "iyilestirme_onerileri.csv"
)

# Karar destek sisteminde kullanılan temel ayarlar hazırlanıyor.
karar_sistemi_ayarlari = {
    "nominal_batarya_kapasitesi_wh": (
        NOMINAL_BATARYA_KAPASITESI_WH
    ),
    "guvenli_batarya_rezervi_yuzde": (
        GUVENLI_BATARYA_REZERVI_YUZDE
    ),
    "kritik_batarya_rezervi_yuzde": (
        KRITIK_BATARYA_REZERVI_YUZDE
    ),
    "model_belirsizlik_payi_wh": (
        MODEL_BELIRSIZLIK_PAYI_WH
    ),
    "batarya_sagligi_araligi": {
        "alt_sinir": BATARYA_SAGLIGI_ALT_SINIRI,
        "ust_sinir": BATARYA_SAGLIGI_UST_SINIRI
    },
    "baslangic_batarya_araligi": {
        "alt_sinir": BASLANGIC_BATARYA_ALT_SINIRI,
        "ust_sinir": BASLANGIC_BATARYA_UST_SINIRI
    },
    "model_girdi_sinirlari": (
        model_girdi_sinirlari
    ),
    "karar_kurallari": {
        "uygun": (
            "Kalan batarya oranı %20 veya daha fazla"
        ),
        "riskli": (
            "Kalan batarya oranı %10 ile %20 arasında"
        ),
        "uygun_degil": (
            "Kalan batarya oranı %10'dan az "
            "veya kullanılabilir enerji yetersiz"
        ),
        "kapsam_disi": (
            "Model eğitim aralığı dışında otomatik onay verilmez"
        )
    },
    "model_sha256": hesaplanan_model_sha256,
    "sistem_notu": (
        "Bu sistem sentetik verilerle geliştirilen bir "
        "kavram kanıtlama çalışmasıdır."
    )
}

# Sistem ayarları JSON dosyasına kaydediliyor.
with open(
    KARAR_SISTEMI_AYAR_DOSYASI,
    "w",
    encoding="utf-8"
) as ayar_dosyasi:
    json.dump(
        karar_sistemi_ayarlari,
        ayar_dosyasi,
        ensure_ascii=False,
        indent=4
    )

# Dört kontrollü test senaryosunun sonuçları kaydediliyor.
senaryo_sonuclari.to_csv(
    SENARYO_SONUCLARI_DOSYASI,
    index=False,
    encoding="utf-8-sig"
)

# Zorlu görev için üretilen öneriler tabloya dönüştürülüyor.
if len(zor_gorev_onerileri["oneriler"]) > 0:
    kaydedilecek_oneriler = pd.DataFrame(
        zor_gorev_onerileri["oneriler"]
    )
else:
    kaydedilecek_oneriler = pd.DataFrame(
        columns=[
            "Öneri Türü",
            "Mevcut Değer",
            "Önerilen Değer",
            "Beklenen Kalan Batarya",
            "Beklenen Durum"
        ]
    )

# Zorlu görevin mevcut durumu öneri dosyasına ek bilgi olarak ekleniyor.
kaydedilecek_oneriler.insert(
    0,
    "Başlangıç Görev Durumu",
    zor_gorev_sonucu["gorev_durumu"]
)

kaydedilecek_oneriler.insert(
    1,
    "Başlangıç Karar Enerjisi (Wh)",
    round(
        zor_gorev_sonucu["karar_enerjisi_wh"],
        2
    )
)

kaydedilecek_oneriler.to_csv(
    IYILESTIRME_ONERILERI_DOSYASI,
    index=False,
    encoding="utf-8-sig"
)

# Kaydedilen CSV dosyaları yeniden yüklenerek doğrulanıyor.
kaydedilen_senaryolar = pd.read_csv(
    SENARYO_SONUCLARI_DOSYASI,
    encoding="utf-8-sig"
)

kaydedilen_oneriler = pd.read_csv(
    IYILESTIRME_ONERILERI_DOSYASI,
    encoding="utf-8-sig"
)

# Kaydedilen karar sırasının beklenen kararlarla aynı olduğu kontrol ediliyor.
kaydedilen_kararlar = (
    kaydedilen_senaryolar[
        "Nihai Onay"
    ].tolist()
)

kayit_dogrulandi = (
    kaydedilen_kararlar
    == beklenen_kararlar
    and len(kaydedilen_senaryolar) == 4
    and KARAR_SISTEMI_AYAR_DOSYASI.exists()
)

if not kayit_dogrulandi:
    raise ValueError(
        "Karar destek sistemi sonuçlarının kayıt doğrulaması başarısız oldu."
    )

print("KARAR DESTEK SİSTEMİ SONUÇLARI BAŞARIYLA KAYDEDİLDİ")
print("=" * 95)
print(
    f"Sistem ayarları       : "
    f"{KARAR_SISTEMI_AYAR_DOSYASI.name}"
)
print(
    f"Senaryo sonuçları     : "
    f"{SENARYO_SONUCLARI_DOSYASI.name}"
)
print(
    f"İyileştirme önerileri : "
    f"{IYILESTIRME_ONERILERI_DOSYASI.name}"
)
print(
    f"Kaydedilen senaryo    : "
    f"{len(kaydedilen_senaryolar)}"
)
print(
    f"Kaydedilen öneri      : "
    f"{len(kaydedilen_oneriler)}"
)
print(
    f"Senaryo testleri      : "
    f"{senaryo_testleri_basarili}"
)
print(
    f"Kayıt doğrulandı      : "
    f"{kayit_dogrulandi}"
)

print("\nKAYDEDİLEN NİHAİ KARARLAR")
print("=" * 95)
display(
    kaydedilen_senaryolar[
        [
            "Senaryo",
            "Görev Durumu",
            "Nihai Onay",
            "Model Kapsamında"
        ]
    ]
)


KARAR DESTEK SİSTEMİ SONUÇLARI BAŞARIYLA KAYDEDİLDİ
Sistem ayarları       : karar_destek_sistemi_ayarlari.json
Senaryo sonuçları     : karar_destek_senaryo_sonuclari.csv
İyileştirme önerileri : iyilestirme_onerileri.csv
Kaydedilen senaryo    : 4
Kaydedilen öneri      : 1
Senaryo testleri      : True
Kayıt doğrulandı      : True

KAYDEDİLEN NİHAİ KARARLAR


,Senaryo,Görev Durumu,Nihai Onay,Model Kapsamında
0,Uygun Görev,Uygun,Uygun,True
1,Riskli Görev,Riskli,Riskli,True
2,Uygun Olmayan Görev,Uygun Değil,Uygun Değil,True
3,Model Kapsamı Dışı,Uygun,Manuel İnceleme Gerekli,False


## 7. Sonuç

Bu çalışmada, daha önce geliştirilen makine öğrenmesi modeli kullanılarak Droneport destekli İHA görevleri için enerji ve batarya temelli bir karar destek sistemi oluşturulmuştur.

Sistem tarafından görev mesafesi, ortalama uçuş hızı, operasyon süresi, faydalı yük ağırlığı, rüzgâr hızı, ortam sıcaklığı ve batarya sıcaklığı doğrulanmış; toplam görev süresi otomatik olarak hesaplanmıştır. Fiziksel veya mantıksal olarak geçersiz girdilerin işlenmesi engellenmiş, modelin eğitim kapsamı dışındaki değerler için güvenilirlik uyarıları oluşturulmuştur.

Görev enerjisi, optimize edilmiş gradyan artırma modeliyle tahmin edilmiştir. Karar sürecinin daha temkinli yürütülmesi amacıyla modelin bağımsız test kümesindeki 12,3851 Wh RMSE değeri tahmine ek belirsizlik payı olarak dahil edilmiştir. Bu yaklaşım istatistiksel bir güvenlik garantisi olarak değil, kavram kanıtlama sistemine ait koruyucu bir varsayım olarak kullanılmıştır.

Nominal batarya kapasitesi 600 Wh olarak kabul edilmiş ve batarya sağlık oranına göre etkin kapasite hesaplanmıştır. Başlangıç batarya seviyesi kullanılarak mevcut enerji belirlenmiş, görev sonrasında kalması beklenen batarya oranı hesaplanmış ve görevler aşağıdaki durumlarla sınıflandırılmıştır:

- **Uygun:** Kalan batarya oranı %20 veya daha fazla,
- **Riskli:** Kalan batarya oranı %10 ile %20 arasında,
- **Uygun Değil:** Kalan batarya oranı %10'un altında veya mevcut enerji yetersiz,
- **Manuel İnceleme Gerekli:** Görev girdilerinden en az biri modelin eğitim kapsamı dışında.

Riskli veya uygun olmayan görevler için başlangıç batarya seviyesinin artırılması, faydalı yükün azaltılması, daha düşük rüzgâr hızının beklenmesi, görev mesafesinin azaltılması ve daha sağlıklı batarya kullanılması seçenekleri bağımsız senaryolar hâlinde araştırılmıştır.

Zorlu görev örneğinde karar enerjisi 556,99 Wh, kullanılabilir enerji ise 382,50 Wh olarak hesaplanmış ve görev uygun bulunmamıştır. Başlangıç batarya seviyesinin %129,21 olması gerektiği belirlendiğinden, yalnızca batarya doluluk oranının artırılması uygulanabilir bir çözüm oluşturmamıştır. Tek değişkenli senaryo aramasında görev mesafesinin 14,00 km'den 3,50 km'ye düşürülmesiyle kalan batarya oranı %23,53 seviyesine yükselmiş ve görev uygun duruma getirilmiştir.

Karar kurallarının doğrulanması amacıyla dört kontrollü senaryo test edilmiştir. Güvenli görev `Uygun`, düşük rezervli görev `Riskli`, kritik rezervli görev `Uygun Değil` olarak sınıflandırılmıştır. Eğitim kapsamının dışındaki 24 km mesafeli görev için ise batarya hesabı uygun görünmesine rağmen otomatik onay verilmemiş ve `Manuel İnceleme Gerekli` sonucu oluşturulmuştur. Bütün senaryo testleri başarıyla tamamlanmıştır.

Sistem ayarları, senaryo sonuçları ve iyileştirme önerileri kalıcı olarak kaydedilmiştir. Böylece veri üretimi, veri temizleme, makine öğrenmesiyle enerji tahmini, hata analizi, görev uygunluk değerlendirmesi ve senaryo tabanlı optimizasyon aşamalarından oluşan kavram kanıtlama projesi tamamlanmıştır.

Geliştirilen sistem sentetik veriler ve varsayımsal güvenlik eşikleri kullanmaktadır. Gerçek bir İHA veya Droneport sisteminde kullanılması için gerçek cihaz verileriyle yeniden eğitilmesi, farklı çevresel koşullarda doğrulanması, üretici sınırlarının uygulanması ve uçuş güvenliği testlerinden geçirilmesi gerekmektedir.
